In [ ]:
#use in terminal py -m pip install openrouteservice geopandas folium shapely

In [7]:
import openrouteservice
from openrouteservice import convert
import folium
import geopandas as gpd
from shapely.geometry import Point

# API KEY de OpenRouteService = ORS
api_key = "eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6ImQ2YzNmOWRhY2UwMjQ0NjhiMGYxNDNhNjBmNGNkOTFiIiwiaCI6Im11cm11cjY0In0="

# Cliente de ORS
client = openrouteservice.Client(key=api_key)

In [8]:
# Coordenadas en formato (longitud, latitud)
location = (-100.3831, 20.5888)  # Ejemplo: Querétaro, México

In [9]:
# Tiempo en segundos (15 minutos)
isochrone = client.isochrones(
    locations=[location],
    profile='driving-car',
    range=[15 * 60],  # 15 minutos
    attributes=['total_pop']  # opcional
)

# Crear un mapa interactivo
m = folium.Map(location=location[::-1], zoom_start=12)
folium.GeoJson(isochrone, name='15 min en coche').add_to(m)
folium.Marker(location[::-1], tooltip="Punto base").add_to(m)
m

In [10]:
# Distancia en metros (por ejemplo, 5 km)
isochrone_km = client.isochrones(
    locations=[location],
    profile='driving-car',
    range=[5000],  # 5 km
    range_type='distance'
)

m2 = folium.Map(location=location[::-1], zoom_start=12)
folium.GeoJson(isochrone_km, name='5 km en coche').add_to(m2)
folium.Marker(location[::-1], tooltip="Punto base").add_to(m2)
m2

In [11]:
from shapely.geometry import Point
import geopandas as gpd

# Crear punto y convertirlo en GeoDataFrame
gdf = gpd.GeoDataFrame(
    {'geometry': [Point(location)]},
    crs='EPSG:4326'  # WGS 84
)

# Convertimos a proyección métrica para aplicar buffer (ej: UTM zona 14N para Querétaro)
gdf_proj = gdf.to_crs(epsg=32614)
gdf_proj['buffer'] = gdf_proj.buffer(5000)  # 5 km buffer
buffer = gdf_proj.set_geometry('buffer').to_crs(epsg=4326)  # Regresamos a lat/lon

# Mostrar en mapa
m3 = folium.Map(location=location[::-1], zoom_start=12)
folium.GeoJson(buffer.geometry.iloc[0], name='Radio 5 km circular').add_to(m3)
folium.Marker(location[::-1], tooltip="Punto base").add_to(m3)
m3
